### ЗАДАЧА: Реестр товаров 

В этой задаче нужно собрать простой каталог товаров для внутренней системы закупок.
Данные приходят как строки, часть строк может быть некорректной.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Класс `Product`:
   - поля `id`, `name`, `_price`, `category`
   - `@property price` (только чтение)
   - метод `set_price(value) -> bool`:
     - возвращает `True`, если цена валидна и обновлена
     - возвращает `False`, если цена невалидна
   - `@classmethod from_line(cls, raw)`:
     - если строка корректная, вернуть `(product, "")`
     - если ошибка, вернуть `(None, "текст причины")`.

2. Класс `ProductRegistry`:
   - хранит список товаров
   - метод `add(product) -> (bool, str)`:
     - `(True, "")`, если успешно
     - `(False, "...")`, если `id` уже существует
   - метод `count()`
   - метод `all_products()`
   - метод `has_id(id)`
   - метод `by_category()` -> `{category: count}`
   - метод `avg_price()`.

3. Загрузка данных:
   - пройти по `lines`
   - использовать `Product.from_line(...)`
   - если товар создан, пытаться `registry.add(...)`
   - все проблемные строки складывать в `problems` как `(line, reason)`.

4. Вывод:
   - число валидных товаров
   - статистика по категориям
   - средняя цена
   - список проблемных строк.


- Для проверки дублей хранить set из `id`.
- В `avg_price()` вернуть `0`, если товаров нет.
- Для проверки, что строка число, можно использовать вспомогательную функцию с `str.replace('.', '', 1).isdigit()`.


In [1]:
lines = [
    "P-1;Mouse;25;Periphery",
    "P-2;Keyboard;45;Periphery",
    "P-3;Cloud VM;120;Services",
    "P-2;Duplicate;50;Periphery",
    "P-4;Broken;-5;Services",
    "P-5;BadPrice;abc;Services",
]


def is_number(text):
    # TODO: вернуть True, если text можно считать числом (включая дроби и знак), иначе False
    if not text:
        return False
    cleaned = text.lstrip('-')
    return cleaned.replace('.', '', 1).isdigit() and cleaned.count('.') <= 1


class Product:
    # TODO: __init__(id, name, price, category)
    def __init__(self, id, name, price, category):
        self.id = id
        self.name = name
        self._price = price
        self.category = category

    # TODO: property price (только getter)
    @property
    def price(self):
        return self._price

    # TODO: set_price(value) -> bool
    def set_price(self,value):
        if is_number(str(value)) and float(value) >=0:
            self._price = float(value)
            return True
        return False

    # TODO: classmethod from_line(cls, raw) -> (product_or_none, reason)
    @classmethod
    def from_line(cls,raw):
        parts = raw.split(';')
        if len(parts) != 4:
            return None, "неккоректное количество"
        
        product_id,name,price_str,category = parts

        if not is_number(price_str):
            return None, f"Цена ,'{price_str}'не является числом"
        
        price = float(price_str)
        if price < 0:
            return None, f"Отрицательная цена: {price}"
        try:
            product = cls( product_id,name,price,category)
            return product,""
        except Exception as e:
            return None, str(e)

    
  
class ProductRegistry:
    # TODO: __init__ (products list + id set)
    def __init__(self):
        self.products = []
        self.ids = set()

    # TODO: add(product) -> (bool, reason)
    def add(self,product):
        if product.id in self.ids:
            return False, f"ID '{product.id} уже существует"
        self.products.append(product)
        self.ids.add(product.id)
        return True,""
    
    # TODO: count()
    def count(self):
        return len(self.products)
    
    # TODO: all_products()
    def all_products(self):
        return self.products
    
    # TODO: has_id(id)
    def has_id(self,id):
        return id in self.ids

    # TODO: by_category()
    def by_category(self):
        categories = {}
        for product in self.products:
            categories[product.category] = categories.get(product.category,0) + 1
        return categories
    
    # TODO: avg_price()
    def avg_price(self):
        if not self.products:
            return 0
        total = sum(product.price for product in self.products)
        return total/ len(self.products)

# TODO: создать registry и problems
registry = ProductRegistry()
problems = []

# TODO: пройти по lines, загрузить валидные товары
for line in lines:
    product,reason = Product.from_line(line)
    if product is not None:
        succes,add_reason = registry.add(product)

# TODO: проблемные строки добавлять в problems как (line, reason)
        if not succes:
            problems.append((line,add_reason))
    else:
        problems.append((line,reason))

# TODO: вывести count, by_category, avg_price и problems
print(f"Количество валидных товаров: {registry.count()}")
print(f"Статистика по категориям: {registry.by_category()}")
print(f"Средняя цена: {registry.avg_price():.2f}")
print("Проблемные строки:")
for line, reason in problems:
    print(f"  '{line}' — {reason}")



Количество валидных товаров: 3
Статистика по категориям: {'Periphery': 2, 'Services': 1}
Средняя цена: 63.33
Проблемные строки:
  'P-2;Duplicate;50;Periphery' — ID 'P-2 уже существует
  'P-4;Broken;-5;Services' — Отрицательная цена: -5.0
  'P-5;BadPrice;abc;Services' — Цена ,'abc'не является числом
